
# Random Forest Regressor Pipeline (Notebook Version)

This notebook replicates the core steps from `torch_prep_kfold.py` **without** K-folds, focused on a clear, linear pipeline:

1. Load two CSVs and merge on a key.
2. Train/test split **by sequence** (regression).
3. Keep **last 90%** of rows per sequence (based on temporal order: sequence → run → frame/time if available).
4. Average numeric features in groups of `navg=280` **per sequence**.
5. Standardize features using **train mean/std** (label unaffected); apply the same transform to test.
6. Train a **RandomForestRegressor** and evaluate (MSE, MAE, R²).
7. Save the fitted model & scaler.

> ⚙️ **You must configure the paths and column names in the first cell below.**


In [2]:
import os, sys, logging
import numpy as np
import pandas as pd

from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from joblib import dump

os.makedirs(OUTPUT_DIR, exist_ok=True)
logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")

In [4]:

# --- CONFIG ---
DATA_FEATURES_CSV = "rawdat.csv"          # computed/processed features
DATA_LABELS_CSV   = "exp_data_all.csv"    # experimental labels
OUTPUT_DIR        = "outputs"             # where to save artifacts
# OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Column names (edit to match your files)
ID_COL      = "sequence"       # key column present in BOTH CSVs to merge on
LABEL_COL   = "bind_avg"     # the regression target
RUN_COL     = "run"          # optional temporal column 1 (e.g., run index)
FRAME_COLS  = []  # optional temporal column 2 (first found is used)

# Split / filtering / aggregation
TEST_SIZE        = 0.2       # fraction of sequences for test split
RANDOM_STATE     = 42
KEEP_LAST_PCT    = 90.0      # keep last X% per sequence
NAVG             = 280       # average numeric features in chunks of N rows per sequence

# Model
N_ESTIMATORS     = 300
MAX_DEPTH        = 10
N_JOBS           = -1

MAX_FEATURES     = "sqrt"
MIN_SAMPLES_SPLIT = 50        
MIN_SAMPLES_LEAF  = 20 

# Metrics to show
from pprint import pprint
print("Config loaded:")
pprint({
    "features_csv": DATA_FEATURES_CSV,
    "labels_csv": DATA_LABELS_CSV,
    "id_col": ID_COL,
    "label_col": LABEL_COL,
    "run_col": RUN_COL,
    "frame_cols": FRAME_COLS,
    "test_size": TEST_SIZE,
    "random_state": RANDOM_STATE,
    "keep_last_pct": KEEP_LAST_PCT,
    "navg": NAVG,
    "rf_params": {"n_estimators": N_ESTIMATORS, "max_depth": MAX_DEPTH, "n_jobs": N_JOBS},
})


Config loaded:
{'features_csv': 'rawdat.csv',
 'frame_cols': [],
 'id_col': 'sequence',
 'keep_last_pct': 90.0,
 'label_col': 'bind_avg',
 'labels_csv': 'exp_data_all.csv',
 'navg': 280,
 'random_state': 42,
 'rf_params': {'max_depth': 10, 'n_estimators': 300, 'n_jobs': -1},
 'run_col': 'run',
 'test_size': 0.2}


In [5]:


def load_csv(filename: str, usecols=None) -> pd.DataFrame:
    df = pd.read_csv(filename, usecols=usecols)
    logging.info(f"Loaded {filename}: shape={df.shape}")
    return df

def require_columns(df: pd.DataFrame, required):
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"Missing required columns {missing}. Available: {df.columns.tolist()}")

def find_first_existing(df: pd.DataFrame, candidates):
    for c in candidates:
        if c in df.columns:
            return c
    return None

def keep_last_n_percent(df: pd.DataFrame, seq_col: str, keep_percent: float,
                        run_col: str=None, frame_col: str=None) -> pd.DataFrame:
    """Keep the last keep_percent% rows per sequence based on temporal sort.
    Sort key: seq_col -> (run_col if present) -> (frame_col if present)."""
    if keep_percent <= 0 or keep_percent >= 100:
        return df.copy()

    sort_cols = [seq_col]
    if run_col and run_col in df.columns:
        sort_cols.append(run_col)
    if frame_col and frame_col in df.columns:
        sort_cols.append(frame_col)

    if len(sort_cols) > 1:
        df_sorted = df.sort_values(sort_cols, kind="mergesort")
    else:
        # fall back to current order if no temporal columns
        df_sorted = df.copy()

    group_sizes = df_sorted.groupby(seq_col)[seq_col].transform("size")
    cumcount = df_sorted.groupby(seq_col).cumcount()
    n_keep = (group_sizes * (keep_percent / 100.0)).astype(int)
    n_keep = n_keep.mask(n_keep < 1, 1)  # ensure at least one if >0%
    mask = cumcount >= (group_sizes - n_keep)
    return df_sorted[mask].reset_index(drop=True)

def average_in_chunks_per_sequence(df: pd.DataFrame, seq_col: str, chunk_size: int,
                                   label_col: str, non_numeric_keep=None) -> pd.DataFrame:
    """Average numeric columns over non-overlapping windows of size chunk_size per sequence.
    - Keeps label by averaging within the chunk (or you can choose median).
    - Keeps non-numeric / identifier columns by taking the first value in the chunk.
    - Drops partial trailing chunks smaller than chunk_size to keep shapes consistent."""
    if non_numeric_keep is None:
        non_numeric_keep = []

    dfs = []
    for seq, sub in df.groupby(seq_col, sort=False):
        n = len(sub)
        n_full = (n // chunk_size) * chunk_size
        if n_full == 0:
            continue
        sub = sub.iloc[:n_full].copy()

        # Add a chunk index within the sequence
        idx = np.arange(n_full) // chunk_size
        sub["_chunk"] = idx

        numeric_cols = sub.select_dtypes(include=[np.number]).columns.tolist()
        # Ensure label is included in numeric averaging if numeric
        if label_col in sub.columns and label_col not in numeric_cols:
            # label might be non-numeric -> try coerce to numeric (optional)
            try:
                sub[label_col] = pd.to_numeric(sub[label_col], errors="coerce")
            except Exception:
                pass
            numeric_cols = sub.select_dtypes(include=[np.number]).columns.tolist()

        # Group-aggregate
        agg_numeric = sub.groupby([seq_col, "_chunk"])[numeric_cols].mean()

        # For non-numeric/id columns we want to keep, take the first row of each chunk
        keep_cols = list({seq_col, label_col, *non_numeric_keep} & set(sub.columns))
        agg_first = sub.groupby([seq_col, "_chunk"])[keep_cols].first()

        out = pd.concat([agg_first, agg_numeric], axis=1)
        # If duplicates due to overlap, numeric cols will duplicate; resolve by grouping again
        out = out.groupby(level=[0,1]).first().reset_index(drop=True)
        dfs.append(out)

    if not dfs:
        return pd.DataFrame()

    result = pd.concat(dfs, axis=0, ignore_index=True)

    # If label_col is duplicated (from numeric+first), keep numeric-averaged version
    if label_col in result.columns:
        # Remove possible duplicate columns with suffixes; ensure single label column
        result = result.loc[:, ~result.columns.duplicated()]

    return result

def standardize_train_test(X_train: pd.DataFrame, X_test: pd.DataFrame):
    scaler = StandardScaler(with_mean=True, with_std=True)
    scaler.fit(X_train)
    X_train_s = pd.DataFrame(scaler.transform(X_train), columns=X_train.columns, index=X_train.index)
    X_test_s  = pd.DataFrame(scaler.transform(X_test),  columns=X_test.columns,  index=X_test.index)
    return scaler, X_train_s, X_test_s



## 1) Load CSVs and Merge
Both CSVs must contain the key column (`ID_COL`). The merged DataFrame must include the target (`LABEL_COL`).


In [6]:

feat_df = load_csv(DATA_FEATURES_CSV)
lab_df  = load_csv(DATA_LABELS_CSV)

require_columns(feat_df, [ID_COL])
require_columns(lab_df,  [ID_COL, LABEL_COL])

df = feat_df.merge(lab_df[[ID_COL, LABEL_COL]], on=ID_COL, how="inner")
logging.info(f"Merged shape: {df.shape}")
display(df.head())


INFO: Loaded rawdat.csv: shape=(272160, 10)
INFO: Loaded exp_data_all.csv: shape=(168, 4)
INFO: Merged shape: (68040, 11)


,sequence,run,VDWAALS,EEL,EGB,ESURF,HB Energy,Hydrophobic Energy,Pi-Pi Energy,Delta_Entropy,bind_avg
0,CAGGGCTGGGTCCACCTCATGGCCTTTGTTCTGGAA,9,-236.997,-1869.660,1823.216,-35.292,-2.590101,-156.445725,-4.282747,-24.750849,0.166339
1,CAGGGCTGGGTCCACCTCATGGCCTTTGTTCTGGAA,9,-218.620,-1850.331,1807.831,-32.521,-2.977171,-142.709472,-7.240534,-25.235404,0.166339
2,CAGGGCTGGGTCCACCTCATGGCCTTTGTTCTGGAA,9,-232.611,-1878.075,1834.181,-34.170,-3.105868,-145.088977,-8.856276,-25.124940,0.166339
3,CAGGGCTGGGTCCACCTCATGGCCTTTGTTCTGGAA,9,-203.677,-1870.595,1823.641,-32.402,-3.414769,-150.961716,-5.338670,-23.079573,0.166339
4,CAGGGCTGGGTCCACCTCATGGCCTTTGTTCTGGAA,9,-212.279,-1864.730,1820.462,-31.858,-3.571942,-146.583284,-7.171679,-22.812241,0.166339



## 2) Train/Test Split by Sequence
We split on **unique sequences** to avoid leakage (all rows of a given sequence go to either train or test).


In [7]:

# Unique sequence ids for grouping
seq_ids = df[ID_COL].values

gss = GroupShuffleSplit(n_splits=1, test_size=TEST_SIZE, random_state=RANDOM_STATE)
train_idx, test_idx = next(gss.split(df, groups=seq_ids))

df_train = df.iloc[train_idx].copy()
df_test  = df.iloc[test_idx].copy()

logging.info(f"Train shape: {df_train.shape} | Test shape: {df_test.shape}")


INFO: Train shape: (53460, 11) | Test shape: (14580, 11)



## 3) Keep Last 90% per Sequence
Temporal ordering uses `ID_COL → RUN_COL → first existing FRAME_COL` (if present). If no temporal column is found, current order is used.


In [8]:

frame_col = find_first_existing(df, FRAME_COLS)
logging.info(f"Temporal columns: run={RUN_COL if RUN_COL in df.columns else None}, frame={frame_col}")

df_train_f = keep_last_n_percent(df_train, seq_col=ID_COL, keep_percent=KEEP_LAST_PCT,
                                 run_col=RUN_COL if RUN_COL in df.columns else None,
                                 frame_col=frame_col)
df_test_f  = keep_last_n_percent(df_test,  seq_col=ID_COL, keep_percent=KEEP_LAST_PCT,
                                 run_col=RUN_COL if RUN_COL in df.columns else None,
                                 frame_col=frame_col)

logging.info(f"After keep_last {KEEP_LAST_PCT}% -> Train: {df_train_f.shape} | Test: {df_test_f.shape}")


INFO: Temporal columns: run=run, frame=None
INFO: After keep_last 90.0% -> Train: (48114, 11) | Test: (13122, 11)



## 4) Average Numeric Columns in Chunks of `navg=280` per Sequence
We average **numeric features** (and label if numeric) in non-overlapping windows. Partial trailing windows are dropped to keep shapes consistent.


In [9]:

# Choose non-numeric columns you want to preserve per chunk (optional)
NON_NUMERIC_KEEP = [ID_COL, RUN_COL] if RUN_COL in df.columns else [ID_COL]

df_train_avg = average_in_chunks_per_sequence(df_train_f, seq_col=ID_COL, chunk_size=NAVG,
                                              label_col=LABEL_COL, non_numeric_keep=NON_NUMERIC_KEEP)
df_test_avg  = average_in_chunks_per_sequence(df_test_f,  seq_col=ID_COL, chunk_size=NAVG,
                                              label_col=LABEL_COL, non_numeric_keep=NON_NUMERIC_KEEP)

logging.info(f"Averaged -> Train: {df_train_avg.shape} | Test: {df_test_avg.shape}")
display(df_train_avg.head())


INFO: Averaged -> Train: (165, 12) | Test: (45, 12)


,run,bind_avg,sequence,VDWAALS,EEL,EGB,ESURF,HB Energy,Hydrophobic Energy,Pi-Pi Energy,Delta_Entropy,_chunk
0,3,1.472119,AACCACTCGACTGACCTCGTGGTCAAATTCCTTACT,-210.803704,-1858.847150,1813.967454,-31.835475,-12.918473,-138.981319,-3.740706,-22.374073,0.0
1,6,1.472119,AACCACTCGACTGACCTCGTGGTCAAATTCCTTACT,-211.512664,-1891.048975,1843.578539,-32.174200,-14.525990,-141.385502,-2.040244,-22.668950,1.0
2,9,1.472119,AACCACTCGACTGACCTCGTGGTCAAATTCCTTACT,-204.243464,-1880.372671,1833.764621,-31.398150,-13.104157,-135.800485,-2.491429,-21.682009,2.0
3,13,1.472119,AACCACTCGACTGACCTCGTGGTCAAATTCCTTACT,-209.674175,-1888.088061,1841.122221,-31.837721,-13.628000,-140.738677,-2.021656,-22.782214,3.0
4,16,1.472119,AACCACTCGACTGACCTCGTGGTCAAATTCCTTACT,-208.561221,-1894.514364,1848.282196,-31.945611,-12.860238,-138.410960,-3.180965,-22.729402,4.0



## 5) Standardize Features (Train Mean/Std) — Label Unchanged
We compute mean/std on **train** features only, then transform both train and test for unbiased evaluation.


In [10]:

# Build X, y
def split_X_y(df_xy: pd.DataFrame, label_col: str):
    require_columns(df_xy, [label_col])
    y = df_xy[label_col].copy()
    # Drop non-feature columns: label + sequence + any obvious meta columns
    drop_cols = {label_col, ID_COL}
    if RUN_COL in df_xy.columns:
        drop_cols.add(RUN_COL)
    # Remove non-numeric columns from features automatically
    X = df_xy.drop(columns=list(drop_cols), errors="ignore")
    X = X.select_dtypes(include=[np.number])
    return X, y

X_train, y_train = split_X_y(df_train_avg, LABEL_COL)
X_test,  y_test  = split_X_y(df_test_avg,  LABEL_COL)

logging.info(f"Feature shapes -> X_train: {X_train.shape}, X_test: {X_test.shape}")

scaler, X_train_s, X_test_s = standardize_train_test(X_train, X_test)


INFO: Feature shapes -> X_train: (165, 9), X_test: (45, 9)



## 6) Train RandomForestRegressor and Evaluate
Metrics: **MSE**, **MAE**, **R²**.


In [11]:

rf = RandomForestRegressor(
    n_estimators=N_ESTIMATORS,
    max_depth=MAX_DEPTH,
    max_features=MAX_FEATURES,         # <- added
    min_samples_split=MIN_SAMPLES_SPLIT,  # <- added
    min_samples_leaf=MIN_SAMPLES_LEAF,    # <- added
    n_jobs=N_JOBS,
    random_state=RANDOM_STATE,
)


rf.fit(X_train_s, y_train)
pred_train = rf.predict(X_train_s)
pred_test  = rf.predict(X_test_s)

metrics = {
    "train": {
        "MSE": mean_squared_error(y_train, pred_train),
        "MAE": mean_absolute_error(y_train, pred_train),
        "R2":  r2_score(y_train, pred_train),
    },
    "test": {
        "MSE": mean_squared_error(y_test, pred_test),
        "MAE": mean_absolute_error(y_test, pred_test),
        "R2":  r2_score(y_test, pred_test),
    }
}
print("Metrics:")
pprint(metrics)


Metrics:
{'test': {'MAE': 0.4717065127724079,
          'MSE': 0.2978687710849478,
          'R2': 0.4214793543534445},
 'train': {'MAE': 0.5841101754172815,
           'MSE': 0.5151639727781564,
           'R2': 0.30020490399959676}}



## 7) Save Artifacts (Model & Scaler)
Saves `.joblib` files for reuse.


In [ ]:

model_path  = os.path.join(OUTPUT_DIR, "rf_regressor.joblib")
scaler_path = os.path.join(OUTPUT_DIR, "scaler.joblib")

dump(rf, model_path)
dump(scaler, scaler_path)

print("Saved:")
print(model_path)
print(scaler_path)



## Notes & Tips

- **Temporal order**: We sort by `seq_id → run → frame/time` if available. Ensure your CSVs have these columns; otherwise, define an appropriate temporal key.
- **Per-sequence vs per-run**: This notebook keeps last 90% **per sequence**. If you want last 90% **per run**, group by `[seq_id, run]` when computing sizes/cumcount.
- **Averaging (`navg`)**: Non-overlapping windows; trailing partial chunks are dropped.
- **Standardization**: Fit on **train** only; transform both train and test.
- **Label column**: Not standardized.
- **Hyperparameters**: Adjust `N_ESTIMATORS`, `MAX_DEPTH` as needed. For reproducibility, keep `RANDOM_STATE` fixed.
- **Leakage guard**: The split is done **by sequence** so test sequences were never seen during training.
